In [ ]:
import pandas as pd
import os

# ---------------------------------------------------------------------
# 1. FUNGSI-FUNGSI UTAMA
# ---------------------------------------------------------------------

def muat_data(path_csv: str) -> pd.DataFrame:
    """Membaca file CSV dan mengubah kolom Timestamp menjadi tipe datetime."""
    df = pd.read_csv(path_csv)
    # Pastikan kolom Timestamp ada sebelum diubah formatnya
    if "Timestamp" not in df.columns:
        raise ValueError("Kolom 'Timestamp' tidak ditemukan di dalam file CSV.")
    df["Timestamp"] = pd.to_datetime(df["Timestamp"])
    return df

def daftar_lokasi(df: pd.DataFrame) -> list:
    """Mengembalikan daftar lokasi unik yang tersedia di dalam data."""
    if "Lokasi" not in df.columns:
        raise ValueError("Kolom 'Lokasi' tidak ditemukan di dalam file CSV.")
    return sorted(df["Lokasi"].unique())

def daftar_kolom_parameter(df: pd.DataFrame) -> list:
    """Mengembalikan daftar kolom parameter (di luar Timestamp dan Lokasi)."""
    return [k for k in df.columns if k not in ("Timestamp", "Lokasi")]

def filter_lokasi(df: pd.DataFrame, lokasi_terpilih: list) -> pd.DataFrame:
    """Menyeleksi baris berdasarkan satu atau beberapa lokasi."""
    return df[df["Lokasi"].isin(lokasi_terpilih)].reset_index(drop=True)

def filter_waktu(df: pd.DataFrame, waktu_mulai: str, waktu_selesai: str) -> pd.DataFrame:
    """Menyeleksi baris berdasarkan rentang waktu (inklusif)."""
    mulai = pd.to_datetime(waktu_mulai)
    selesai = pd.to_datetime(waktu_selesai)
    mask = (df["Timestamp"] >= mulai) & (df["Timestamp"] <= selesai)
    return df[mask].reset_index(drop=True)

def pilih_kolom(df: pd.DataFrame, kolom_terpilih: list) -> pd.DataFrame:
    """Menyeleksi kolom yang ingin ditampilkan."""
    kolom_wajib = ["Timestamp", "Lokasi"]
    kolom_final = kolom_wajib + [k for k in kolom_terpilih if k not in kolom_wajib]
    return df[kolom_final]

# ---------------------------------------------------------------------
# 2. MODE INTERAKTIF & INPUT PENGGUNA
# ---------------------------------------------------------------------

def _pilih_dari_daftar(prompt: str, opsi: list) -> list:
    """Menampilkan daftar bernomor dan meminta pengguna memilih."""
    print(f"\n{prompt}")
    for i, o in enumerate(opsi, start=1):
        print(f"  {i}. {o}")
    jawaban = input("Pilih nomor (pisah koma, atau ketik 'semua'): ").strip().lower()

    if jawaban == "semua" or jawaban == "":
        return opsi

    indeks = [int(x.strip()) - 1 for x in jawaban.split(",") if x.strip().isdigit()]
    return [opsi[i] for i in indeks if 0 <= i < len(opsi)]

def main():
    print("=== Program Seleksi Data CSV ===")

    # --- PROSES INPUT FILE CSV YANG DIPERBARUI ---
    df = None
    while True:
        path_csv = input("\nMasukkan path/nama file CSV (contoh: data_saya.csv atau C:/folder/data.csv): ").strip()

        if not path_csv:
            print("Path file tidak boleh kosong. Silakan masukkan nama file.")
            continue

        if not os.path.exists(path_csv):
            print(f"Error: File '{path_csv}' tidak ditemukan di sistem. Periksa kembali lokasi dan nama filenya.")
            continue

        try:
            df = muat_data(path_csv)
            print(f"\n[Berhasil] Data dimuat: {len(df)} baris.")
            print(f"Rentang waktu data: {df['Timestamp'].min()} s.d. {df['Timestamp'].max()}")
            break # Keluar dari loop jika file berhasil dibaca
        except Exception as e:
            print(f"Gagal memproses file CSV: {e}")
            print("Pastikan file tersebut berformat CSV dan memiliki kolom 'Timestamp' serta 'Lokasi'.")

    # --- pilih lokasi ---
    lokasi_terpilih = _pilih_dari_daftar("Pilih lokasi yang ingin diseleksi:", daftar_lokasi(df))
    df = filter_lokasi(df, lokasi_terpilih)

    # --- pilih rentang waktu ---
    print("\nRentang waktu data saat ini:", df["Timestamp"].min(), "s.d.", df["Timestamp"].max())
    mulai = input("Waktu mulai (kosongkan untuk memakai batas paling awal): ").strip()
    selesai = input("Waktu selesai (kosongkan untuk memakai batas paling akhir): ").strip()
    mulai = mulai if mulai else str(df["Timestamp"].min())
    selesai = selesai if selesai else str(df["Timestamp"].max())
    df = filter_waktu(df, mulai, selesai)

    # --- pilih kolom parameter ---
    kolom_terpilih = _pilih_dari_daftar("Pilih parameter yang ingin ditampilkan:", daftar_kolom_parameter(df))
    df = pilih_kolom(df, kolom_terpilih)

    # --- ringkasan & simpan ---
    print(f"\nHasil seleksi: {len(df)} baris, {len(df.columns)} kolom.")
    print(df.head(10))

    simpan = input("\nSimpan hasil ke file CSV baru? (y/n) [y]: ").strip().lower()
    if simpan in ("", "y", "ya"):
        nama_output = input("Nama file output [hasil_seleksi.csv]: ").strip()
        nama_output = nama_output if nama_output else "hasil_seleksi.csv"
        df.to_csv(nama_output, index=False)
        print(f"Data tersimpan di: {nama_output}")

if __name__ == "__main__":
    main()

=== Program Seleksi Data CSV ===

Masukkan path/nama file CSV (contoh: data_saya.csv atau C:/folder/data.csv): /content/dummy_data_flisr_1bulan.csv

[Berhasil] Data dimuat: 60480 baris.
Rentang waktu data: 2026-08-01 00:00:00 s.d. 2026-08-30 23:55:00

Pilih lokasi yang ingin diseleksi:
  1. GD 01
  2. GD 02
  3. GD 03
  4. GD 04
  5. GD 05
  6. GI 1
  7. GI 2
Pilih nomor (pisah koma, atau ketik 'semua'): 1

Rentang waktu data saat ini: 2026-08-01 00:00:00 s.d. 2026-08-30 23:55:00
Waktu mulai (kosongkan untuk memakai batas paling awal): 
Waktu selesai (kosongkan untuk memakai batas paling akhir): 

Pilih parameter yang ingin ditampilkan:
  1. Ia (A)
  2. Ib (A)
  3. Ic (A)
  4. Va (V)
  5. Vb (V)
  6. Vc (V)
  7. VL (V)
  8. IL (A)
  9. PF
  10. Pa (kW)
  11. Pb (kW)
  12. Pc (kW)
  13. Pt (kW)
  14. Sa (kVA)
  15. Sb (kVA)
  16. Sc (kVA)
  17. St (kVA)
  18. Cos_a
  19. Cos_b
  20. Cos_c
  21. Sin_a
  22. Sin_b
  23. Sin_c
  24. Tan_a
  25. Tan_b
  26. Tan_c
Pilih nomor (pisah koma, at

In [ ]:
"""
pilih_data_gardu.py
========================
Program untuk membaca file CSV dan menyeleksi baris data
HANYA berdasarkan lokasi/gardu yang dipilih. Seluruh kolom parameter
akan otomatis disertakan.

Cara penggunaan:
Jalankan langsung dari terminal:
    python pilih_data_gardu.py
"""

import pandas as pd
import os

# ---------------------------------------------------------------------
# 1. FUNGSI-FUNGSI UTAMA
# ---------------------------------------------------------------------

def muat_data(path_csv: str) -> pd.DataFrame:
    """Membaca file CSV dan memvalidasi formatnya."""
    df = pd.read_csv(path_csv)

    # Validasi keberadaan kolom esensial
    if "Timestamp" not in df.columns:
        raise ValueError("Kolom 'Timestamp' tidak ditemukan di dalam file CSV.")
    if "Lokasi" not in df.columns:
        raise ValueError("Kolom 'Lokasi' tidak ditemukan di dalam file CSV.")

    df["Timestamp"] = pd.to_datetime(df["Timestamp"])
    return df

def daftar_lokasi(df: pd.DataFrame) -> list:
    """Mengembalikan daftar lokasi (gardu) unik yang tersedia di dalam data."""
    return sorted(df["Lokasi"].unique())

def filter_lokasi(df: pd.DataFrame, lokasi_terpilih: list) -> pd.DataFrame:
    """Menyeleksi baris berdasarkan satu atau beberapa lokasi/gardu."""
    return df[df["Lokasi"].isin(lokasi_terpilih)].reset_index(drop=True)


# ---------------------------------------------------------------------
# 2. MODE INTERAKTIF & INPUT PENGGUNA
# ---------------------------------------------------------------------

def _pilih_dari_daftar(prompt: str, opsi: list) -> list:
    """Menampilkan daftar bernomor dan meminta pengguna memilih lokasi."""
    print(f"\n{prompt}")
    for i, o in enumerate(opsi, start=1):
        print(f"  {i}. {o}")
    jawaban = input("Pilih nomor (pisah koma, atau ketik 'semua'): ").strip().lower()

    if jawaban == "semua" or jawaban == "":
        return opsi

    indeks = [int(x.strip()) - 1 for x in jawaban.split(",") if x.strip().isdigit()]
    return [opsi[i] for i in indeks if 0 <= i < len(opsi)]

def main():
    print("=== Program Seleksi Data CSV (Berdasarkan Gardu) ===")

    # --- PROSES INPUT FILE CSV ---
    df = None
    while True:
        path_csv = input("\nMasukkan path/nama file CSV (contoh: data.csv atau C:/folder/data.csv): ").strip()

        if not path_csv:
            print("Path file tidak boleh kosong. Silakan masukkan nama file.")
            continue

        if not os.path.exists(path_csv):
            print(f"Error: File '{path_csv}' tidak ditemukan di sistem. Periksa kembali lokasi dan nama filenya.")
            continue

        try:
            df = muat_data(path_csv)
            print(f"\n[Berhasil] Data dimuat: {len(df)} baris dan {len(df.columns)} kolom.")
            break # Keluar dari loop jika file berhasil dibaca
        except Exception as e:
            print(f"Gagal memproses file CSV: {e}")

    # --- PILIH LOKASI / GARDU ---
    lokasi_terpilih = _pilih_dari_daftar("Pilih Gardu / Lokasi yang ingin diekstrak datanya:", daftar_lokasi(df))
    df = filter_lokasi(df, lokasi_terpilih)

    # --- RINGKASAN & SIMPAN ---
    print(f"\nHasil seleksi: {len(df)} baris, {len(df.columns)} kolom (Seluruh parameter disertakan).")

    # Menampilkan sedikit cuplikan data agar bisa diverifikasi
    print("\nCuplikan 5 baris pertama hasil seleksi:")
    print(df.head(5))

    simpan = input("\nSimpan hasil ke file CSV baru? (y/n) [y]: ").strip().lower()
    if simpan in ("", "y", "ya"):
        nama_output = input("Nama file output [hasil_seleksi_gardu.csv]: ").strip()
        nama_output = nama_output if nama_output else "hasil_seleksi_gardu.csv"
        df.to_csv(nama_output, index=False)
        print(f"Data tersimpan di: {nama_output}")

if __name__ == "__main__":
    main()

=== Program Seleksi Data CSV (Berdasarkan Gardu) ===

Masukkan path/nama file CSV (contoh: data.csv atau C:/folder/data.csv): /content/dummy_data_flisr_1bulan.csv

[Berhasil] Data dimuat: 60480 baris dan 28 kolom.

Pilih Gardu / Lokasi yang ingin diekstrak datanya:
  1. GD 01
  2. GD 02
  3. GD 03
  4. GD 04
  5. GD 05
  6. GI 1
  7. GI 2
Pilih nomor (pisah koma, atau ketik 'semua'): 3

Hasil seleksi: 8640 baris, 28 kolom (Seluruh parameter disertakan).

Cuplikan 5 baris pertama hasil seleksi:
            Timestamp Lokasi  Ia (A)  Ib (A)  Ic (A)   Va (V)   Vb (V)  \
0 2026-08-01 00:00:00  GD 03   54.64   53.91   54.00  11361.6  11396.4   
1 2026-08-01 00:05:00  GD 03   55.43   56.41   57.93  11346.0  11387.6   
2 2026-08-01 00:10:00  GD 03   54.79   53.82   54.01  11460.6  11428.8   
3 2026-08-01 00:15:00  GD 03   57.02   53.64   55.72  11408.8  11458.8   
4 2026-08-01 00:20:00  GD 03   59.06   57.41   57.41  11413.8  11427.2   

    Vc (V)   VL (V)  IL (A)  ...  St (kVA)   Cos_a   Cos